# Лекция 5. Обход графа в глубину: куда можно дойти

> Конспект пятого занятия курса «Алгоритмы и структуры данных» (ДПО).
> В лекции 4 граф научились хранить четырьмя способами. Почти каждый вопрос к нему касался одного ребра или соседей одной вершины.
> Теперь вопрос дальний: куда можно дойти из вершины, если идти по рёбрам сколько угодно шагов.
> Ответ даёт поиск в глубину. Он обходит всё достижимое и ни в одну вершину не заходит дважды.
> Пять задач контеста покажут, что можно узнать таким обходом.

**После лекции вы сможете:**

- увидеть граф в лабиринте из клеток, даже если рёбра нигде не записаны;
- объяснить, как поиск в глубину уходит вглубь и возвращается, и назвать стек вызовов на любом шаге;
- узнать один каркас в обходе по клеткам и в обходе по списку смежности: проверить, пометить, пойти к соседям;
- проверить, связен ли граф, найти его компоненты и решить, дерево ли он;
- объяснить, почему обход стоит `O(|V| + |E|)`.

**Что нужно знать:** лекция 4: список смежности, сумма длин списков равна `2 · |E|`. Лекции 2 и 3: рекурсивная функция. Лекция 3: предел рекурсии в Python из раздела 9. Лекции 2 и 3: дерево. Python: функции, списки, `input()`.

**Время:** 30 минут на чтение. Около двух часов, если запускать код, вводить примеры и делать все предсказания. Удобная пауза после раздела 4. Дальше каждая ячейка самодостаточна, перезапускать ничего не нужно.

**Как читать.** Ячейки запускайте по порядку, сверху вниз. Блоки «Предскажите» и «Попробуйте сами» это барьеры: сначала отвечаете сами, потом открываете спойлер. Спойлеры обведены горизонтальными линиями. Решения задач контеста читают ввод через `input()`, как в Яндекс Контесте. Запустите ячейку и наберите строки примера из условия, по одной, каждую завершая Enter. Ожидаемый ответ записан комментарием в первой строке ячейки, а сам пример стоит над ячейкой блоком, строка к строке. Остальные ячейки выполняются сами.

---

## 1. Заливка: сколько клеток зальёт краска

В графическом редакторе есть инструмент «заливка» (flood fill). Вы щёлкаете по клетке, и краска растекается. Она закрашивает эту клетку, потом соседние, потом соседей соседей. Течёт краска только вверх, вниз, влево и вправо. По диагонали не течёт, через стену не перелезает.

Возьмите картинку попроще, лабиринт. Звёздочка это стена, точка это пустая клетка. Строки и столбцы считаются с нуля.

```
******
*..*.*
*.**.*
*..*.*
**.*.*
******
```

![Лабиринт 6 на 6, точками отмечены клетки (1, 1) и (1, 4)](img/maze.png)

**Предскажите.** Краска падает в клетку `(1, 1)`: строка 1, столбец 1. Сколько клеток она закрасит? А если щёлкнуть по клетке `(1, 4)`? Сколько щелчков нужно, чтобы закрасить все пустые клетки?

---

<details><summary>Ответ</summary>

Из `(1, 1)` краска закрасит 6 клеток: `(1, 1)`, `(1, 2)`, `(2, 1)`, `(3, 1)`, `(3, 2)`, `(4, 2)`. Это левая комната.

Из `(1, 4)` закрасит 4 клетки: правый коридор от `(1, 4)` до `(4, 4)`.

Всех пустых клеток 10, и нужно 2 щелчка. Пустые клетки распались на две комнаты, а из одной в другую краска не попадает.
</details>

---

На картинке 6 на 6 ответ видно глазами. На картинке 10 000 на 10 000 клеток глазами уже не посчитать. Нужна программа, а для неё лабиринт придётся увидеть как граф.

## 2. Лабиринт это граф, который не хранят

Каждая пустая клетка это вершина. Две пустые клетки соединены ребром, если у них общая сторона. Стены в граф не входят. Краска течёт ровно по рёбрам этого графа. Вопрос «сколько клеток закрасит заливка» превращается в вопрос «до скольких вершин можно дойти по рёбрам из стартовой».

В лекции 4 граф хранили одним из четырёх способов. Здесь хранить рёбра не нужно. Такой граф называют **неявным** (implicit graph): рёбра не записаны, их вычисляют по правилу. Соседи клетки `(i, j)` это четыре клетки вокруг неё, их можно вычислить. Сдвиги к ним записывают списком.

In [ ]:
maze = [
    "******",
    "*..*.*",
    "*.**.*",
    "*..*.*",
    "**.*.*",
    "******",
]

steps = [[0, 1], [1, 0], [0, -1], [-1, 0]]   # вправо, вниз, влево, вверх


def cell_neighbours(i, j):
    result = []
    for di, dj in steps:
        if maze[i + di][j + dj] == '.':        # сосед пустой: есть ребро
            result.append((i + di, j + dj))
    return result


print(cell_neighbours(1, 1))
print(cell_neighbours(3, 2))
print(cell_neighbours(1, 4))

Пара `[di, dj]` это сдвиг по строке и по столбцу. `[0, 1]` это шаг вправо, `[1, 0]` вниз, `[0, -1]` влево, `[-1, 0]` вверх. Соседи выходят в порядке сдвигов. У клетки `(3, 2)` справа стена, поэтому первым идёт нижний сосед `(4, 2)`, а за ним левый `(3, 1)`.

Проверки границ здесь нет. Лабиринт обнесён стенами по краю, поэтому у любой пустой клетки все четыре соседа существуют, и индекс не выходит за пределы списка. В задаче A4 ниже это гарантирует условие.

Граф есть. Осталось придумать, как обойти все вершины, до которых можно дойти, и ни в одну не зайти дважды.

## 3. Поиск в глубину: идти вглубь, потом возвращаться

Правило обхода такое. Зайдите в клетку и пометьте её. Потом по очереди попробуйте каждого соседа. Если сосед пустой и не помечен, зайдите в него по тому же правилу. Когда соседи кончились, вернитесь туда, откуда пришли, и продолжайте перебирать соседей там.

Обход уходит по одному пути как можно дальше. Возвращается он, только когда идти некуда. Поэтому его называют **поиском в глубину** (depth-first search, DFS). Вы его уже встречали: обход дерева по порядку из лекции 3 тоже уходит вглубь до `None` и возвращается.

«Зайти по тому же правилу» в коде означает вызвать ту же функцию. Поэтому поиск в глубину удобно писать рекурсией.

In [ ]:
field = [list(row) for row in maze]   # копия: обход перекрашивает клетки
cnt = 0
order = []                            # только для трассы: порядок посещения


def dfs(i: int, j: int):
    global cnt
    if field[i][j] == '*':            # стена или уже посещённая клетка
        return
    field[i][j] = '*'                 # пометили: сюда больше не зайдём
    cnt += 1
    order.append((i, j))
    for di, dj in steps:
        dfs(i + di, j + dj)


dfs(1, 1)
print(cnt)

Три детали этого кода.

- **Посещённая клетка становится стеной.** Отдельный список пометок не нужен. В стену обход не заходит, значит, и в посещённую клетку тоже.
- **Проверка стоит при входе в функцию.** Функция `cell_neighbours` здесь не нужна. `dfs` вызывается для всех четырёх соседей и только потом смотрит, можно ли сюда. Поэтому вызовов больше, чем клеток. Каждая из 6 клеток вызывает `dfs` четыре раза, плюс первый вызов: всего 25 вызовов. Из них 19 сразу возвращаются.
- **`global cnt` здесь нужен.** Строка `cnt += 1` присваивает переменной новое значение. Без `global` Python решит, что `cnt` своя переменная функции, и упадёт с `UnboundLocalError`. Списки `field` и `order` функция меняет на месте, а не присваивает заново, поэтому для них `global` не нужен. Правило короткое: читать глобальную переменную и менять её на месте можно без `global`, он нужен только для присваивания.

Как обход возвращается назад? Каждый вызов `dfs` ждёт, пока закончатся вызовы, которые он сделал сам. Пока он ждёт, он помнит, на каком соседе остановился его цикл `for`. Когда вложенный вызов закончится, цикл продолжит со следующего соседа. Цепочку незаконченных вызовов называют **стеком вызовов** (call stack). В лабиринте её удобно видеть как путь от стартовой клетки до текущей.

**Предскажите.** Соседи перебираются в порядке `steps`: вправо, вниз, влево, вверх. Из какой клетки обход зайдёт в `(2, 1)`? Сколько клеток будет на стеке вызовов в самый глубокий момент?

In [ ]:
print(order)

---

<details><summary>Ответ</summary>

В `(2, 1)` обход зайдёт из `(1, 1)`. Сначала `(1, 1)` уйдёт вправо, в `(1, 2)`. Оттуда идти некуда, вызов закончится, и цикл в `(1, 1)` возьмёт следующего соседа, нижнего.

Самый длинный стек из 5 клеток: `(1, 1)`, `(2, 1)`, `(3, 1)`, `(3, 2)`, `(4, 2)`. На миг к ним добавляется шестой вызов, для стены рядом с `(4, 2)`, но он сразу возвращается.

Порядок посещения: `(1, 1)`, `(1, 2)`, `(2, 1)`, `(3, 1)`, `(3, 2)`, `(4, 2)`.

| шаг | что происходит | стек вызовов, от старта к текущей клетке |
|---|---|---|
| 1 | зашли в `(1, 1)` | `(1, 1)` |
| 2 | справа пусто: зашли в `(1, 2)` | `(1, 1)` `(1, 2)` |
| 3 | у `(1, 2)` справа, снизу и сверху стены, слева посещённая `(1, 1)`. Возврат | `(1, 1)` |
| 4 | у `(1, 1)` следующий сосед снизу: зашли в `(2, 1)` | `(1, 1)` `(2, 1)` |
| 5 | справа стена, снизу пусто: зашли в `(3, 1)` | `(1, 1)` `(2, 1)` `(3, 1)` |
| 6 | справа пусто: зашли в `(3, 2)` | `(1, 1)` `(2, 1)` `(3, 1)` `(3, 2)` |
| 7 | справа стена, снизу пусто: зашли в `(4, 2)` | `(1, 1)` `(2, 1)` `(3, 1)` `(3, 2)` `(4, 2)` |
| 8 | у `(4, 2)` справа, снизу и слева стены, сверху посещённая `(3, 2)`. Возврат в `(3, 2)` | `(1, 1)` `(2, 1)` `(3, 1)` `(3, 2)` |
| 9 | у `(3, 2)` слева посещённая `(3, 1)`, сверху стена. Возврат в `(3, 1)` | `(1, 1)` `(2, 1)` `(3, 1)` |
| 10 | у `(3, 1)` снизу и слева стены, сверху посещённая `(2, 1)`. Возврат в `(2, 1)` | `(1, 1)` `(2, 1)` |
| 11 | у `(2, 1)` слева стена, сверху посещённая `(1, 1)`. Возврат в `(1, 1)` | `(1, 1)` |
| 12 | у `(1, 1)` слева и сверху стены. Обход закончен | пусто |

На шаге 3 видна суть поиска в глубину. Из `(1, 2)` идти некуда, и обход отступает на шаг назад. Там цикл берёт следующего соседа по списку. На шагах 8–12 обход отступает до самого старта: у каждой клетки на пути непроверенные соседи уже стены.

Анимация: синие и оранжевая клетки это стек вызовов, оранжевая клетка текущая, его верхушка. Зелёные клетки обход уже закончил.

![Поиск в глубину по лабиринту](img/fill.gif)
</details>

---

### Задача A4: площадь комнаты

> **A4. Площадь комнаты.** Требуется вычислить площадь комнаты в квадратном лабиринте.
> **Ввод.** В первой строке вводится число N – размер лабиринта (3 <= N <= 10). В следующих N строках задан лабиринт (‘.’ – пустая клетка, ‘*’ – стенка). И наконец, последняя строка содержит два числа – номер строки и столбца клетки, находящейся в комнате, площадь которой необходимо вычислить. Гарантируется, что эта клетка пустая и что лабиринт окружен стенками со всех сторон.
> **Вывод.** Требуется вывести единственное число – количество пустых клеток в данной комнате.
> **Ограничения.** 1 секунда, 64 МБ.

В условии два примера. В первом комната из одной клетки, возьмём второй. Ввод:

```
4
****
*..*
*..*
****
3 3
```

Вывод:

```
4
```

Это та же функция `dfs`, что выше. Добавилось только чтение ввода. Список `field` сначала заполняется пустыми списками-заготовками, потом каждая заменяется строкой лабиринта, разбитой на символы. В условии строки и столбцы считаются с единицы. `lambda x: int(x) - 1` это короткая функция «превратить в число и вычесть единицу», и `map` применяет её к обоим номерам.

In [ ]:
# введите 6 строк из примера 2; ответ: 4
n = int(input())
field = [[] for _ in range(n)]

for i in range(n):
    field[i] = list(input())

i_start, j_start = map(lambda x: int(x) - 1, input().split())

steps = [  # i + di, j + dj
    [0, 1], [1, 0], [0, -1], [-1, 0]
]

cnt = 0
def dfs(i: int, j: int):
    global cnt
    if field[i][j] == '*':
        return
    field[i][j] = '*'
    cnt += 1
    for di, dj in steps:
        dfs(i + di, j + dj)
dfs(i_start, j_start)
print(cnt)

Клетки здесь ни при чём. Обходу нужно одно: для каждой вершины знать её соседей. А это умеет список смежности из лекции 4.

## 4. Тот же обход на списке смежности

Вот граф из пяти вершин. Рёбра `0-1`, `0-2`, `1-3`, `2-3`, `2-4`.

![Граф из пяти вершин и пяти рёбер](img/graph.png)

Его список смежности: у вершины `0` соседи `1` и `2`, у вершины `1` соседи `0` и `3`, и так далее. Имя `adj` в этой лекции означает список смежности. В лекции 4 так называлась матрица смежности, а список назывался `neighbours`. В коде всех задач ниже `adj` это список.

In [ ]:
adj = [[1, 2], [0, 3], [0, 3, 4], [1, 2], [2]]   # соседи вершин 0, 1, 2, 3, 4
used = [0] * 5                                   # 1, если вершина посещена
order = []                                       # только для трассы


def dfs(v: int) -> None:
    used[v] = 1
    order.append(v)
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)

Каркас тот же, что в лабиринте. Сравните строки.

| шаг каркаса | лабиринт, A4 | список смежности |
|---|---|---|
| проверить, можно ли зайти | `if field[i][j] == '*': return` при входе в функцию | `if used[to] == 0:` перед вызовом |
| пометить | `field[i][j] = '*'` | `used[v] = 1` |
| пойти к соседям | `for di, dj in steps: dfs(i + di, j + dj)` | `for to in adj[v]: dfs(to)` |

Отличается только место проверки. В лабиринте функция вызывается и сама отказывается входить. Здесь вызов делается, только если вершина ещё не посещена. Результат одинаковый, а лишних вызовов во втором варианте нет.

`global` в этом коде не нужен по той же причине, что для `field` в разделе 3: `used[v] = 1` меняет список на месте.

**Предскажите.** В каком порядке `dfs(0)` посетит вершины? У вершины `0` два соседа, `1` и `2`. Сколько раз цикл в `dfs(0)` вызовет `dfs`?

In [ ]:
dfs(0)
print(order)

---

<details><summary>Ответ</summary>

Порядок: `0`, `1`, `3`, `2`, `4`. Цикл в `dfs(0)` вызовет `dfs` один раз, для вершины `1`.

| вызов | сосед `to` | `used[to]` | действие |
|---|---|---|---|
| `dfs(0)` | 1 | 0 | вызвать `dfs(1)` |
| `dfs(1)` | 0 | 1 | пропустить |
| `dfs(1)` | 3 | 0 | вызвать `dfs(3)` |
| `dfs(3)` | 1 | 1 | пропустить |
| `dfs(3)` | 2 | 0 | вызвать `dfs(2)` |
| `dfs(2)` | 0 | 1 | пропустить |
| `dfs(2)` | 3 | 1 | пропустить |
| `dfs(2)` | 4 | 0 | вызвать `dfs(4)` |
| `dfs(4)` | 2 | 1 | пропустить. Соседи кончились, возврат в `dfs(2)` |
| `dfs(2)` | | | соседи кончились, возврат в `dfs(3)` |
| `dfs(3)` | | | соседи кончились, возврат в `dfs(1)` |
| `dfs(1)` | | | соседи кончились, возврат в `dfs(0)` |
| `dfs(0)` | 2 | 1 | пропустить: вершину `2` уже посетили через `3`. Обход закончен |

Вершина `2` соседняя с `0`, но обход пришёл в неё длинным путём `0 → 1 → 3 → 2`. Он ушёл вглубь по первому соседу и успел обойти всё, что за ним. Когда очередь дошла до второго соседа `0`, тот был уже помечен.

В самый глубокий момент на стеке пять вызовов: `dfs(0)`, `dfs(1)`, `dfs(3)`, `dfs(2)`, `dfs(4)`.

Анимация: синие и оранжевая вершины это стек вызовов, оранжевая вершина текущая, его верхушка. Синие рёбра ведут по стеку от старта. Зелёные вершины обход уже закончил.

![Поиск в глубину по графу](img/dfs.gif)
</details>

---

Что можно узнать одним обходом? Начнём с самого простого вопроса: дошёл ли обход до всех вершин.

## 5. Задача B4: связность

Граф называется **связным** (connected), если из любой вершины можно дойти по рёбрам до любой другой.

**Попробуйте сами, прежде чем читать дальше.** Проверять все пары вершин долго. Почему для неориентированного графа достаточно одного обхода из вершины `1`?

---

<details><summary>Ответ</summary>

Пусть обход из `1` дошёл до всех вершин. Возьмите любые две вершины `u` и `w`. Из `1` есть путь в `u`. Рёбра неориентированные, поэтому тот же путь проходится обратно, из `u` в `1`. Из `1` есть путь в `w`. Склейте: `u → 1 → w`.

Если же обход из `1` до какой-то вершины не дошёл, то пути от `1` до неё нет, и граф не связный.
</details>

---

> **B4. Связность.** В этой задаче требуется проверить, что граф является связным, то есть что из любой вершины можно по рёбрам этого графа попасть в любую другую. В графе могут существовать петли и кратные ребра.
> **Ввод.** В первой строке входного файла заданы числа `N` и `M` через пробел — количество вершин и рёбер в графе, соответственно (`1 ⩽ N ⩽ 100`, `0 ⩽ M ⩽ 10 000`). Следующие `M` строк содержат по два числа `u_i` и `v_i` через пробел (`1 ⩽ u_i, v_i ⩽ N`); каждая такая строка означает, что в графе существует ребро между вершинами `u_i` и `v_i`.
> **Вывод.** Выведите “YES”, если граф является связным, и “NO” в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример 1 из условия. Ввод:

```
3 2
1 2
3 2
```

Вывод:

```
YES
```

Пример 2 из условия. Ввод:

```
3 1
1 3
```

Вывод:

```
NO
```

In [ ]:
# введите 3 строки из примера 1; ответ: YES
n, m = map(int, input().split())

adj = [[] for _ in range(n + 1)]

for _ in range(m):
    v, u = map(int, input().split())
    adj[v].append(u)
    adj[u].append(v)

used = [0] * (n + 1)

def dfs(v: int) -> None:
    used[v] = 1
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)

dfs(1)

cnt = sum(used)
if cnt == n:
    print("YES")
else:
    print("NO")

Здесь вершины нумеруются с единицы, как во вводе. Списки заведены длиной `n + 1`, и ячейка `0` просто не используется. Номер из ввода сразу служит индексом, вычитать единицу не нужно. В лекции 4 делали наоборот: сдвигали номер на `i - 1`. Оба способа рабочие. Ячейка `used[0]` так и остаётся нулём, поэтому `sum(used)` считает ровно посещённые вершины.

Петли и кратные рёбра решению не мешают. Петля кладёт вершину в её собственный список, но к этому моменту вершина уже помечена. Второе такое же ребро ведёт в уже помеченную вершину.

Если ответ NO, хочется знать больше. На сколько кусков распался граф и какая вершина в каком?

## 6. Задача C4: компоненты связности

**Компонента связности** (connected component) это кусок графа, внутри которого из любой вершины можно дойти до любой. Расширить его нельзя: из него нет рёбер наружу. Обход из вершины помечает ровно её компоненту, не больше и не меньше.

Вернитесь к заливке из раздела 1. Одна заливка закрашивает одну комнату, то есть одну компоненту. Вопрос «сколько щелчков нужно» это вопрос «сколько в графе компонент».

Как их найти: перебрать вершины по порядку. Если вершина ещё не помечена, она открывает новую компоненту. Увеличьте счётчик `color` и запустите из неё обход. Обход пометит всю компоненту номером `color`. Следующие вершины этой компоненты цикл пропустит: они уже помечены.

![Граф из девяти вершин](img/components_plain.png)

**Предскажите.** У вершины `9` нет ни одного ребра. Будет ли она компонентой? Какой номер компоненты получит вершина `6`?

In [ ]:
n = 9
edge_list = [(1, 2), (2, 3), (1, 3), (4, 5), (6, 7), (7, 8)]

adj = [[] for _ in range(n + 1)]
for v, u in edge_list:
    adj[v].append(u)
    adj[u].append(v)

used = [0] * (n + 1)
color = 0

def dfs(v: int) -> None:
    used[v] = color
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)

for v in range(1, n + 1):
    if used[v] == 0:
        color += 1
        dfs(v)

print(color)
print(*used[1:])

---

<details><summary>Ответ</summary>

Компонент четыре: `{1, 2, 3}`, `{4, 5}`, `{6, 7, 8}` и `{9}`. Вершина без рёбер тоже компонента, из одной вершины. Цикл дойдёт до `9`, увидит, что она не помечена, и откроет для неё номер 4.

Вершина `6` получит номер 3. Номера раздаются в порядке, в котором цикл встречает непомеченные вершины: `1`, `4`, `6`, `9`.

![Тот же граф, вершины раскрашены по компонентам](img/components.png)
</details>

---

Обратите внимание на две детали. Список `used` теперь хранит не 0 или 1, а номер компоненты: 0 значит «ещё не посещена». А `color` внутри `dfs` только читается, поэтому `global` для него не нужен.

Это решение задачи C4 с данными, записанными прямо в коде.

> **C4. Компоненты связности.** Дан неориентированный невзвешенный граф. Необходимо посчитать количество его компонент связности и вывести их.
> **Ввод.** Во входном файле записано два числа N и M (0 < N <= 100000, 0 <= M <= 100000). В следующих M строках записаны по два числа i и j (1 <= i, j <= N), которые означают, что вершины i и j соединены ребром.
> **Вывод.** В первой строчке выходного файла выведите количество компонент связности. Далее выведите `N` целых чисел, `i`-е из них задаёт номер компоненты связности для `i`-й вершины. Компоненты следует нумеровать последовательными целыми числами от 1. Порядок нумерации компонент произвольный.
> **Ограничения.** 1 секунда, 512 МБ.

Пример из условия. Ввод:

```
4 2
1 2
3 4
```

Вывод:

```
2
1 1 2 2
```

In [ ]:
# введите 3 строки из примера; ответ: 2 и 1 1 2 2
import sys
sys.setrecursionlimit(1000000)

n, m = map(int, input().split())
adj = [[] for _ in range(n + 1)]

for _ in range(m):
    v, u = map(int, input().split())
    adj[v].append(u)
    adj[u].append(v)

used = [0] * (n + 1)
color = 0

def dfs(v: int) -> None:
    used[v] = color
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)

for v in range(1, n + 1):
    if used[v] == 0:
        color += 1
        dfs(v)

print(color)
print(*used[1:])

Первые две строки поднимают **предел рекурсии** (recursion limit). Это тот самый лимит из раздела 9 лекции 3: Python разрешает около тысячи вложенных вызовов. В C4 вершин до 100 000. Если граф это цепочка `1-2-3-…`, обход уйдёт вглубь на всю её длину, и каждый шаг вглубь это ещё один вложенный вызов. С поднятым пределом цепочка из 100 000 вершин обходится примерно за 0,2 секунды.

**Цена обхода.** Каждая вершина помечается один раз, поэтому `dfs` для неё вызывается один раз. Внутри цикл один раз проходит её список соседей. Сумма длин всех списков равна `2 · |E|`, это вы считали в разделе 8 лекции 4. Значит, весь обход стоит `O(|V| + |E|)`: столько же, сколько памяти занимает сам список смежности. Цикл по всем компонентам в C4 стоит столько же. Обходы разных компонент не пересекаются, и каждая вершина попадает ровно в один.

В лабиринте вершины это клетки, а у каждой клетки не больше четырёх соседей. Обход лабиринта `N` на `N` стоит `O(N²)`.

Компоненты считать вы умеете. По их числу и числу рёбер можно узнать форму графа.

## 7. Задача D4: дерево ли это

**Цикл** (cycle) это маршрут по рёбрам, который возвращается в начальную вершину и не проходит ни одно ребро дважды. Например, треугольник `1-2-3-1`. Не путайте с циклом `for` в коде. **Дерево** это связный граф без циклов. В лекциях 2 и 3 у дерева был корень. Здесь корня нет, но форма та же: от любой вершины до любой другой ведёт ровно один путь.

Проверять циклы напрямую не придётся. Хватит компонент и числа рёбер. Представьте, что граф собирается по одному ребру. Вначале `n` вершин без рёбер, это `n` компонент. Каждое новое ребро делает одно из двух:

- соединяет две разные компоненты, и компонент становится на одну меньше;
- соединяет две вершины одной компоненты. Между ними уже был путь, и вместе с новым ребром он замыкается в цикл.

Чтобы из `n` компонент осталась одна, нужно `n − 1` соединяющих рёбер. Пусть граф связный и рёбер ровно `n − 1`. Соединяющих среди них нужно не меньше `n − 1`, значит, соединяющие все. Ни одно ребро не замкнуло цикл. Если рёбер больше, хотя бы одно замкнуло цикл.

> **D4. Снова дерево!** Дан неориентированный граф из N вершин и M ребер. Необходимо проверить, является ли граф деревом. Дерево — это связный граф без циклов.
> **Ввод.** Во входном файле в первой строке содержатся два целых числа N и M (1 ≤ N ≤ 100, 0 ≤ M ≤ 1000), записанные через пробел. Далее следуют M различных строк с описаниями ребер, каждая из которых содержит два натуральных числа A_i и B_i (1 ≤ A_i ≤ B_i ≤ N), где A_i и B_i — номера вершин, соединенных i-м ребром.
> **Вывод.** В выходной файл выведите слово «YES», если граф является деревом, или «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример 1 из условия. Ввод:

```
3 2
1 2
1 3
```

Вывод:

```
YES
```

Пример 2 из условия. Ввод:

```
4 3
1 2
1 3
2 3
```

Вывод:

```
NO
```

Решение это C4 с другим концом: вместо вывода компонент проверка двух условий. Строка `setrecursionlimit` поднимает предел, как в C4, только до 100 000. При `N ≤ 100` она не нужна, но и не мешает. То же в E4.

In [ ]:
# введите 3 строки из примера 1; ответ: YES
import sys
sys.setrecursionlimit(100000)
n, m = map(int, input().split())

adj = [[] for _ in range(n + 1)]

for _ in range(m):
    v, u = map(int, input().split())
    adj[v].append(u)
    adj[u].append(v)

used = [0] * (n + 1)
color = 0

def dfs(v: int) -> None:
    used[v] = color
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)

for v in range(1, n + 1):
    if used[v] == 0:
        color += 1
        dfs(v)

if color == 1 and m == n - 1:
    print("YES")
else:
    print("NO")

**Предскажите.** В примере 2 вершин 4, рёбер 3, то есть `m == n - 1`. Почему ответ всё равно NO? Какое из двух условий не выполнено?

---

<details><summary>Ответ</summary>

Не выполнено `color == 1`. Компонент две: `{1, 2, 3}` и `{4}`. Вершина `4` без рёбер.

Три ребра потрачены на треугольник `1-2-3`. Первые два соединили компоненты, третье замкнуло цикл. На вершину `4` ребра не осталось. Поэтому одного условия `m == n - 1` мало, нужна ещё связность. И наоборот: квадрат `1-2-3-4-1` связный, но в нём 4 ребра при 4 вершинах, и это тоже не дерево.
</details>

---

До сих пор обход отвечал числом или словом YES и NO. В последней задаче ответ это сам маршрут обхода.

## 8. Задача E4: пошаговый обход

Поиск в глубину буквально ходит по графу. Он заходит в соседа по ребру, а закончив с ним, возвращается по тому же ребру. Если записывать каждый шаг, и вперёд, и назад, получится маршрут, который проходит по рёбрам и побывает везде.

> **E4. Пошаговый обход графа.** Пошаговым обходом графа из вершины `v` назовём последовательность вершин `u_1, u_2, …, u_r` такую, что: `u_1 = u_r = v`; каждая вершина графа, достижимая из `v`, встречается в ней хотя бы один раз; между любыми двумя соседними вершинами последовательности в графе существует ребро. Задан связный неориентированный граф и его вершина `v`. Выведите любой пошаговый обход этого графа.
> **Ввод.** В первой строке входного файла заданы числа `N`, `M` и `v` через пробел — количество вершин и рёбер в графе и начальная вершина обхода (`1 ⩽ N ⩽ 100`, `0 ⩽ M ⩽ 10 000`, `1 ⩽ v ⩽ N`). Следующие `M` строк содержат по два числа `u_i` и `v_i` через пробел (`1 ⩽ u_i, v_i ⩽ N`); каждая такая строка означает, что в графе существует ребро между вершинами `u_i` и `v_i`.
> **Вывод.** В первой строке входного файла выведите число `r` — количество вершин в найденном пошаговом обходе (`1 ⩽ r ⩽ 10 000`; гарантируется, что обход, удовлетворяющий этим ограничениям, существует). Во второй строке выведите сами числа `u_1, u_2, …, u_r` через пробел.
> **Ограничения.** 1 секунда, 256 МБ.

Пример 1 из условия. Ввод:

```
3 2 1
1 2
2 3
```

Вывод:

```
5
1 2 3 2 1
```

Пример 2 из условия. Ввод:

```
4 4 1
1 2
2 3
3 4
4 1
```

Вывод:

```
5
1 2 3 4 1
```

Вершина попадает в маршрут по двум поводам. Первый раз при входе в `dfs`. Потом каждый раз, когда обход вернулся к ней из соседа: строка `path.append(v)` стоит после вызова `dfs(to)`. Поэтому вершина, из которой обход ушёл в трёх соседей, встретится в маршруте четыре раза, а вершина без непосещённых соседей один раз.

In [ ]:
# введите 3 строки из примера 1; ответ: 5 и 1 2 3 2 1
import sys
sys.setrecursionlimit(100000)

n, m, start = map(int, input().split())
adj = [[] for _ in range(n + 1)]

for _ in range(m):
    v, u = map(int, input().split())
    adj[v].append(u)
    adj[u].append(v)

used = [0] * (n + 1)
path = []

def dfs(v: int) -> None:
    used[v] = 1
    path.append(v)
    for to in adj[v]:
        if used[to] == 0:
            dfs(to)
            path.append(v)          # вернулись из to обратно в v

dfs(start)
print(len(path))
print(*path)

Запустите ячейку ещё раз и введите пример 2. Решение выведет `7` и `1 2 3 4 3 2 1`, а в условии `5` и `1 2 3 4 1`. Оба ответа правильные: условие просит любой пошаговый обход. Короткий маршрут возвращается в `1` по ребру `4-1`. Поиск в глубину так не умеет, он всегда возвращается той же дорогой, что пришёл.

Теперь введите граф из раздела 4. Номера вершин сдвинуты на единицу, потому что здесь счёт с `1`:

```
5 5 1
1 2
1 3
2 4
3 4
3 5
```

Решение выведет `9` и `1 2 4 3 5 3 4 2 1`. Здесь обход ни разу не отступил, пока не дошёл до вершины `5`. Поэтому первые пять вершин маршрута это порядок посещения из раздела 4, `0 1 3 2 4`, сдвинутый на единицу. Дальше идут возвраты: из `5` в `3`, из `3` в `4`, из `4` в `2`, из `2` в `1`.

**Попробуйте сами, прежде чем читать дальше.** В связном графе из `n` вершин какой длины получается маршрут этого решения? Проверьте ответ на примере 1 и на графе из раздела 4.

---

<details><summary>Ответ</summary>

Ровно `2n − 1`. При входе каждая вершина пишется один раз, это `n` записей. После каждого вызова `dfs`, кроме самого первого, вызвавшая функция добавляет одну запись возврата. Таких вызовов `n − 1`, по одному на каждую вершину, кроме стартовой. Всего `n + (n − 1) = 2n − 1`.

Пример 1: `n = 3`, маршрут из 5 вершин. Граф из раздела 4: `n = 5`, маршрут из 9 вершин. При `n ≤ 100` маршрут не длиннее 199, и ограничение `r ⩽ 10 000` из условия выполнено с запасом.
</details>

---

## 9. Проверьте себя

Ответьте без кода, потом сверьтесь.

1. Лабиринт `5` на `5`: стены по краю, внутри квадрат `3` на `3` из пустых клеток. Обход из A4 стартует в центре. Сколько раз будет вызвана `dfs`?
2. Почему в A4 нет списка `used`, а клетка всё равно не посещается дважды?
3. В графе из раздела 4 список вершины `0` записали в другом порядке: `[2, 1]`. В каком порядке `dfs(0)` посетит вершины?
4. В графе 6 вершин, 5 рёбер и две компоненты связности. Это дерево? Есть ли в нём цикл?
5. Почему в A4 без `global cnt` код падает, а в B4 `dfs` меняет `used` без `global`?

---

<details><summary>Ответы</summary>

1. 37 раз. Пустых клеток 9, каждая вызывает `dfs` для четырёх соседей, это 36 вызовов. Плюс первый вызов из основной программы.
2. Посещённая клетка перекрашивается в `'*'`. Для обхода она становится стеной, а в стену он не заходит.
3. `0`, `2`, `3`, `1`, `4`. Обход уходит в `2`, оттуда в `3`, из `3` в `1`. Из `1` идти некуда, обход возвращается в `3`, потом в `2`, и у `2` остаётся непосещённый сосед `4`.
4. Не дерево: граф не связный. Цикл есть. Из 6 компонент в 2 превращают 4 соединяющих ребра. Значит, одно из пяти рёбер соединило вершины одной компоненты и замкнуло цикл.
5. `cnt += 1` присваивает имени `cnt` новое число, и без `global` Python считает `cnt` переменной функции. `used[v] = 1` меняет содержимое списка, а само имя `used` нового значения не получает.
</details>

---

## 10. Итог

- Граф бывает спрятан в данных. В лабиринте вершины это пустые клетки, а соседей вычисляют сдвигами `steps`. Хранить рёбра не нужно.
- Поиск в глубину заходит в вершину, помечает её и по очереди уходит в непомеченных соседей. Когда идти некуда, он возвращается на шаг назад. Цепочка незаконченных вызовов это стек вызовов, путь от старта до текущей вершины.
- Каркас один для клеток и для списка смежности: проверить, пометить, пойти к соседям. Проверка стоит при входе в функцию или перед вызовом.
- Одним обходом проверяют связность (B4). Циклом по вершинам находят компоненты (C4). Компоненты и число рёбер говорят, дерево ли это (D4). Записанные шаги дают маршрут по графу (E4).
- Обход стоит `O(|V| + |E|)`: каждая вершина посещается один раз, каждый список соседей просматривается один раз. На длинной цепочке рекурсии нужен поднятый предел `sys.setrecursionlimit`.

**Где встретится дальше.** На следующем занятии обход в ширину (breadth-first search, BFS). Он посещает те же вершины, но в другом порядке.

**Что повторить через два-три дня.** Нарисуйте граф из шести вершин с рёбрами `1-2`, `1-3`, `2-4`, `3-4`, `5-6`. Запишите его список смежности. Проведите `dfs(1)` на бумаге: в каком порядке он посетит вершины и как будет расти и убывать стек вызовов. Сколько у графа компонент и почему он не дерево?